# Deploy Your Agent with FastAPI

**What you will build:** the same agent you've been running in cells, exposed as a real **HTTP API** — including a **streaming** endpoint so responses feel instant. This is the code equivalent of n8n's "Deploy to Production": your agent stops being a notebook and becomes a service other software can call.

> **Run it:** open in [Google Colab](https://colab.research.google.com/github/ezponda/ai-agents-course/blob/main/courses/python_code/book/30_deploy_fastapi.ipynb) (nothing to install), or run locally in Jupyter. Each notebook installs what it needs in its first cell.

In [ ]:
# Setup — PydanticAI on OpenRouter. Run once.
!pip install -q "pydantic-ai-slim[openai]>=2.0,<3.0"

import os, getpass
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Paste your OpenRouter API key: ")

MODEL_NAME = "meta-llama/llama-3.3-70b-instruct"
model = OpenAIChatModel(MODEL_NAME, provider=OpenAIProvider(
    base_url="https://openrouter.ai/api/v1", api_key=os.environ["OPENROUTER_API_KEY"]))
print("Ready:", MODEL_NAME)

## First, streaming — because a blocking agent feels broken

An agent that takes 20 seconds to return a wall of text feels frozen. Streaming shows tokens as they're generated, so the user sees it working. PydanticAI streams with `run_stream` (an async context manager) and `stream_text(delta=True)`, which yields each new chunk. Watch it print live:

In [ ]:
agent = Agent(model, system_prompt="You are a concise assistant.")

async with agent.run_stream("Explain what an API is, in 3 short sentences.") as response:
    async for chunk in response.stream_text(delta=True):   # delta=True -> only the NEW text each step
        print(chunk, end="", flush=True)

That token-by-token flow is what every good chat UI does. Now we wrap the agent in a web API that offers both a plain JSON endpoint and a streaming one.

## The API

We write it to a file with `%%writefile` (a notebook can't host a long-running server cleanly — see the note below). In an `async def` route we use `await agent.run(...)` (never `run_sync`, which errors inside a running event loop). The streaming route uses FastAPI's `StreamingResponse` with Server-Sent Events (`text/event-stream`). Two production habits are baked in from the start: the model id comes from the environment (`MODEL_NAME`), and **every route sits behind an API-key check**.

```{warning}
**Never ship a public agent endpoint without auth.** Anyone with the URL can spend your LLM budget. The gate below (`AGENT_API_KEY` + an `X-API-Key` header) is the *minimum* — a real service adds per-user rate limits (3.1), logging, and abuse monitoring. Both `AGENT_API_KEY` and `OPENROUTER_API_KEY` come from the host's environment; never hard-code a secret into `app.py`.
```

In [ ]:
%%writefile app.py
import os
from fastapi import Depends, FastAPI, Header, HTTPException
from fastapi.responses import StreamingResponse
from pydantic import BaseModel
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

MODEL_NAME = os.getenv("MODEL_NAME", "meta-llama/llama-3.3-70b-instruct")   # config from the env
model = OpenAIChatModel(MODEL_NAME, provider=OpenAIProvider(
    base_url="https://openrouter.ai/api/v1", api_key=os.environ["OPENROUTER_API_KEY"]))
agent = Agent(model, system_prompt="You are a concise assistant.")

app = FastAPI()

# --- Auth: the minimum gate for a public endpoint ---
API_KEY = os.getenv("AGENT_API_KEY")     # set in the host's env; if unset the gate is open (dev only)

def require_api_key(x_api_key: str | None = Header(default=None)):
    if API_KEY and x_api_key != API_KEY:
        raise HTTPException(status_code=401, detail="invalid or missing API key")

class Message(BaseModel):
    text: str
    session_id: str = "demo"             # per-user/conversation id (used by the LangGraph version below)

@app.post("/chat", dependencies=[Depends(require_api_key)])          # plain JSON: wait for the full reply
async def chat(msg: Message):
    result = await agent.run(msg.text)
    return {"reply": result.output}

@app.post("/chat/stream", dependencies=[Depends(require_api_key)])   # streaming: tokens as Server-Sent Events
async def chat_stream(msg: Message):
    async def gen():
        async with agent.run_stream(msg.text) as response:
            async for chunk in response.stream_text(delta=True):
                yield f"data: {chunk}\n\n"
    return StreamingResponse(gen(), media_type="text/event-stream")

## Run it

Locally (a normal terminal), install and launch:

```bash
pip install fastapi uvicorn "pydantic-ai-slim[openai]>=2.0,<3.0"
export OPENROUTER_API_KEY=sk-or-...
export AGENT_API_KEY=choose-a-long-random-string   # the gate on your endpoint
uvicorn app:app --reload
```

Then call it (send the key as the `X-API-Key` header):

```bash
curl -X POST localhost:8000/chat -H 'content-type: application/json' \
     -H "X-API-Key: $AGENT_API_KEY" -d '{"text": "Hello!"}'
curl -N -X POST localhost:8000/chat/stream -H 'content-type: application/json' \
     -H "X-API-Key: $AGENT_API_KEY" -d '{"text": "Tell me a joke"}'
```

```{note}
**Running a server inside Colab** is awkward — the notebook already owns the event loop, so `uvicorn.run(...)` blocks, and Colab ports aren't public. For real deployment, run the `.py` on a host (Railway, Render, Fly, a VM) exactly as above. To demo from Colab, use `nest_asyncio` + a background thread + a tunnel (ngrok/cloudflared) — but the script-on-a-host path is what you'll actually ship.
```

## Ship it: a container + a smoke test

Hosts differ, but a container runs anywhere. This is the whole Dockerfile — copy the app, install the pins, run uvicorn on the port the host provides:

In [ ]:
%%writefile Dockerfile
FROM python:3.12-slim
WORKDIR /app
COPY app.py .
RUN pip install --no-cache-dir fastapi uvicorn "pydantic-ai-slim[openai]>=2.0,<3.0"
ENV PORT=8000
CMD ["sh", "-c", "uvicorn app:app --host 0.0.0.0 --port ${PORT}"]

Any container host works — Render, Railway, Fly. On **Render**: *New → Web Service*, point it at this repo/directory, set `OPENROUTER_API_KEY` and `AGENT_API_KEY` in the environment, and let it build from the Dockerfile. The instant it's live, run **one smoke test** — the deploy equivalent of an eval (1.6):

```bash
curl -H "X-API-Key: $AGENT_API_KEY" -H "Content-Type: application/json" \
     -d '{"text":"Say hello","session_id":"smoke-1"}' \
     https://your-service.onrender.com/chat
```

A `200` with a `reply` means wiring, secrets, and auth are all correct. A `401` means your `X-API-Key` doesn't match the server's `AGENT_API_KEY` — the gate is working. Make this curl the first thing you run after every deploy; a green smoke test is the difference between "it deployed" and "it works".

::::{dropdown} 🔍 The LangGraph version
:color: info

A LangGraph agent deploys the same way — `await graph.ainvoke(...)` for JSON, `graph.astream(..., stream_mode="messages")` for streaming. The one thing you **must** get right is the `thread_id`: it selects *whose* conversation the checkpointer (2.3) loads. Take it from the request, one per user or conversation — a hardcoded constant would make every caller share (and overwrite) a single thread, which is exactly the memory-bleed failure 2.3's error table warns about.

```python
class Message(BaseModel):
    text: str
    session_id: str        # the client sends a stable id per user/conversation

@app.post("/chat/stream")
async def stream(msg: Message):
    cfg = {"configurable": {"thread_id": msg.session_id}}   # per-user — NEVER a constant like "web-1"
    async def gen():
        async for chunk, meta in graph.astream(
                {"messages": [{"role": "user", "content": msg.text}]}, cfg, stream_mode="messages"):
            if chunk.content:
                yield f"data: {chunk.content}\n\n"
    return StreamingResponse(gen(), media_type="text/event-stream")
```

```{warning}
`thread_id` is a security boundary, not a formality — it's the server-side twin of the `deps`-based `customer_id` from 1.3. Derive it from your authenticated user (or a signed session), never let one client read another's thread. A shared or client-spoofable `thread_id` leaks conversations across users.
```

LangGraph also has a managed deployment product (LangGraph Platform) if you don't want to run the server yourself.
::::

::::{dropdown} 🔍 One conversation, two requests at once (double texting)
:color: info

The `thread_id` warnings above kept *different users* apart. A subtler case is the *same* user firing a second message before the first finishes — tapping send twice, or a flaky client retrying. Both requests load the same checkpoint (2.3) and both write back: the second run can overwrite the first's state, or their tool side-effects can interleave (two emails sent, a charge applied twice).

Keep two identities separate:

- **User identity** — *who* is calling (from your auth). Concurrency across different users is fine and parallel; their threads never touch.
- **Conversation identity** — *which* `thread_id`. Concurrency *within one thread* is the problem, because that is shared mutable state.

Three ways to handle it, cheapest first:

1. **Serialize per thread.** Hold a lock per `thread_id` so a second request waits (or is rejected) until the first finishes. A dict of `asyncio.Lock`s is enough for a single process:

    ```python
    import asyncio
    from collections import defaultdict
    _locks = defaultdict(asyncio.Lock)          # one lock per thread_id

    async def run_turn(thread_id, text):
        async with _locks[thread_id]:           # same thread waits; other threads run free
            ...                                  # load state, run the agent, save
    ```

2. **Reject or replace the second request.** Return `409 Conflict` while a thread is busy, or cancel the in-flight run and start the new one ("interrupt" semantics). Which you want is a product decision.
3. **Idempotency keys on side-effects.** Give each external action (send email, charge card) a key derived from the request, so a duplicate or retry is a no-op instead of a second charge. This is the only defense that survives crashes and retries, not just concurrency.

Across a fleet of processes a single-process lock isn't enough — move the boundary to your datastore (a row lock or an atomic checkpoint compare-and-set) or a per-thread queue. LangGraph Platform builds this in, with named **double-texting** modes (`reject`, `interrupt`, `rollback`, `enqueue`); the plain FastAPI service above leaves the choice to you. Rule of thumb: **parallel across threads, serialized within a thread — and idempotent wherever the agent touches the outside world** (3.1 adds the per-user rate limit that stops one user opening the floodgate).
::::

**What's next:** in **31** we put a real front end in front of this API — the same move as n8n's "Connect Your App."